# Residual-stream SAEs for LFM2.5-230M

This notebook runs on Kaggle x2 T4 GPUs.

Trains two BatchTopK sparse autoencoders on the residual stream of
`LiquidAI/LFM2.5-230M`

- **GPU 1** -> hook `model.layers.6` (residual stream after layer 6, ~50% depth)
- **GPU 2** -> hook `model.layers.10` (residual stream after layer 10, ~75% depth)

Dataset: FineWeb-Edu (streamed), 1024-token contexts, ~100M tokens.

In [ ]:
SMOKE = False  # True: ~500k-token pipeline test. False: full 100M-token run.

MODEL_NAME = "LiquidAI/LFM2.5-230M"
LAYERS = [(6, 0), (10, 1)]  # (layer_index, gpu_index)
D_MODEL = 1024
EXPANSION = 16
K = 64
CONTEXT_SIZE = 1024
TRAINING_TOKENS = 500_000 if SMOKE else 100_000_000
N_CHECKPOINTS = 0 if SMOKE else 2
WORK = "/kaggle/working"

print(f"SMOKE={SMOKE}  training_tokens={TRAINING_TOKENS:,}")

In [ ]:
%pip install -q -U sae-lens "transformers>=5.4"

In [ ]:
import importlib.metadata as md

import torch

for pkg in ["sae-lens", "transformers", "transformer-lens", "torch", "datasets"]:
    try:
        print(f"{pkg}: {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg}: NOT INSTALLED")
n_gpus = torch.cuda.device_count()
print(f"GPUs: {n_gpus}", [torch.cuda.get_device_name(i) for i in range(n_gpus)])
assert n_gpus >= 2, "This notebook expects the 'GPU T4 x2' accelerator"

## Sanity checks

Before training: load the model and verify that hooking the output of
`model.layers.N` yields exactly `hidden_states[N+1]`, i.e. the residual stream
after layer N.

The model ships in bf16 and T4 has no bf16, so training reads fp16 activations.
Finiteness only rules out overflow, so the check below also measures how far fp16
moves things: relative error on the layer-6 and layer-10 residuals, and the KL
divergence of the next-token distribution against fp32. Those are the numbers that
say whether the SAE is being trained on a faithful copy of the residual stream.

In [ ]:
import gc

from transformers import AutoModelForCausalLM, AutoTokenizer

SAMPLE_TEXT = (
    "The Industrial Revolution began in Britain in the late eighteenth "
    "century and transformed manufacturing, transport, and agriculture. "
    "Steam engines, mechanized looms, and railways reshaped daily life, "
    "while cities grew rapidly around new factories. Historians debate the "
    "degree to which living standards improved for workers during the first "
    "decades of industrialization."
)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
enc = tok(SAMPLE_TEXT, return_tensors="pt").to("cuda:0")


def run_dtype(dtype: torch.dtype):
    """Hidden states and final-token logits under one dtype."""
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to("cuda:0").eval()
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True)
    hs = [h.float().cpu() for h in out.hidden_states]
    logits = out.logits[0, -1].float().cpu()

    captured = {}

    def hook(_mod, _inp, output):
        captured["resid"] = output[0] if isinstance(output, tuple) else output

    handle = model.model.layers[6].register_forward_hook(hook)
    with torch.no_grad():
        out2 = model(**enc, output_hidden_states=True)
    handle.remove()
    assert captured["resid"].shape[-1] == D_MODEL
    assert torch.equal(captured["resid"], out2.hidden_states[7]), "model.layers.6 output != hidden_states[7]"

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return hs, logits


hs32, logits32 = run_dtype(torch.float32)
hs16, logits16 = run_dtype(torch.float16)
print("hook(model.layers.6) == hidden_states[7]  [OK]")

finite = all(torch.isfinite(h).all().item() for h in hs16)
kl = torch.nn.functional.kl_div(
    logits16.log_softmax(-1), logits32.log_softmax(-1), log_target=True, reduction="sum"
).item()
print(f"\nfp16 all hidden states finite: {finite}")
print(f"fp16 vs fp32 next-token KL: {kl:.2e} nats")
for i, (a, b) in enumerate(zip(hs32, hs16)):
    rel = ((a - b).norm(dim=-1) / a.norm(dim=-1).clamp_min(1e-6))[0]
    flag = "  <- hook point" if i in (7, 11) else ""
    print(f"hidden_states[{i:2d}] mean_norm={a.norm(dim=-1).mean():9.2f} rel_err mean={rel.mean():.2e} max={rel.max():.2e}{flag}")

MODEL_DTYPE = "float16" if finite else "float32"
if not finite:
    print("fp16 produced non-finite activations; falling back to fp32")
    assert all(torch.isfinite(h).all().item() for h in hs32), "model is non-finite even in fp32?!"
print(f"\nMODEL_DTYPE = {MODEL_DTYPE}")

## Training entrypoint

Training lives in `train_sae.py`. Run this notebook from a
working directory where that file is present. The launch cell below starts one
process per GPU.

## Launch both trainings

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

script_path = Path("train_sae.py")
assert script_path.exists(), "train_sae.py must be present in the notebook working directory"

procs = []
for layer, gpu in LAYERS:
    env = dict(
        os.environ,
        CUDA_VISIBLE_DEVICES=str(gpu),
        TOKENIZERS_PARALLELISM="false",
        WANDB_MODE="disabled",
    )
    log_path = Path(WORK) / f"train_layer{layer}.log"
    log_file = open(log_path, "w")
    cmd = [
        sys.executable, "train_sae.py",
        "--layer", str(layer),
        "--dtype", MODEL_DTYPE,
        "--training-tokens", str(TRAINING_TOKENS),
        "--n-checkpoints", str(N_CHECKPOINTS),
    ]
    procs.append((layer, subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT, env=env), log_file))
    print(f"launched layer {layer} on GPU {gpu} -> {log_path}")


def tail(path: Path, n: int = 3) -> str:
    try:
        lines = path.read_text(errors="replace").strip().splitlines()
        return "\n".join(lines[-n:])
    except FileNotFoundError:
        return "(no log yet)"


start = time.time()
while any(p.poll() is None for _, p, _ in procs):
    time.sleep(120)
    mins = (time.time() - start) / 60
    print(f"\n===== t+{mins:.0f} min =====")
    for layer, p, _ in procs:
        state = "running" if p.poll() is None else f"exited {p.returncode}"
        print(f"--- layer {layer} [{state}] ---")
        print(tail(Path(WORK) / f"train_layer{layer}.log"))

failed = False
for layer, p, log_file in procs:
    log_file.close()
    print(f"\nlayer {layer}: exit code {p.returncode}")
    if p.returncode != 0:
        failed = True
        print("".join(f"  {ln}\n" for ln in tail(Path(WORK) / f"train_layer{layer}.log", 60).splitlines()))
assert not failed, "at least one training process failed - see logs above"
print(f"\nboth trainings finished in {(time.time() - start) / 3600:.2f} h")

## Eval

Evaluate the trained SAEs from this notebook's output directory.

Explained variance on its own does not say the dictionary is useful, so this also
reports:

- **CE loss recovered** — splice `decode(encode(x))` back into the forward pass and
  measure how much of the cross-entropy gap between the clean model and a
  mean-ablated layer the SAE closes. This is the metric that tracks whether
  downstream computation survives; EV can look high while it is poor.
- **KL vs clean** — same splice, divergence of the next-token distribution.
- **A same-rank PCA baseline** — PCA is variance-optimal, so if a dense rank-L0
  projection matches the SAE's EV then EV is not evidence about the dictionary.
- **Training-time firing density** from `sparsity.safetensors`, because a feature
  can only look dead on an eval set of this size if it is rarer than 1/n_tokens.


## Outputs

In [ ]:
for f in sorted(Path(WORK).rglob("*")):
    if f.is_file():
        print(f"{f.stat().st_size / 1e6:10.1f} MB  {f}")